In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
train_path = 'GiveMeSomeCredit-training.csv'
test_path = 'GiveMeSomeCredit-testing.csv'

df_train = pd.read_csv(train_path).drop(columns=['Unnamed: 0'], errors='ignore')
df_test = pd.read_csv(test_path).drop(columns=['Unnamed: 0'], errors='ignore')

def prepare_data(data, is_train=True):
    # Tratamento de Nulos
    data['MonthlyIncome'] = data['MonthlyIncome'].fillna(data['MonthlyIncome'].median())
    data['NumberOfDependents'] = data['NumberOfDependents'].fillna(0)

    # Engenharia de Variáveis
    data['TotalLate'] = (data['NumberOfTime30-59DaysPastDueNotWorse'] +
                        data['NumberOfTime60-89DaysPastDueNotWorse'] +
                        data['NumberOfTimes90DaysLate'])

    features = ['RevolvingUtilizationOfUnsecuredLines', 'age', 'DebtRatio', 'TotalLate']
    X = data[features].copy()

    if is_train:
        return X, data['SeriousDlqin2yrs']
    return X

X_train_raw, y_train_raw = prepare_data(df_train)
X_test_raw = prepare_data(df_test, is_train=False)

# Escalonamento para o intervalo [0, 1] - Crucial para funções de pertinência Fuzzy
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Conversão para tensores
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_raw.values, dtype=torch.float32).reshape(-1, 1)

# 2. DEFINIÇÃO DO MODELO NEURO-FUZZY
class NeuroFuzzyModel(nn.Module):
    def __init__(self, input_dim, n_rules):
        super(NeuroFuzzyModel, self).__init__()
        self.n_rules = n_rules

        # Inicialização dos centros (mu) distribuídos e larguras (sigma)
        initial_mu = torch.linspace(0, 1, n_rules).repeat(input_dim, 1)
        self.mu = nn.Parameter(initial_mu)
        self.sigma = nn.Parameter(torch.ones(input_dim, n_rules) * 0.2)

        # Camada de consequentes (Regras Lineares)
        self.rule_weights = nn.Linear(n_rules, 1)

    def forward(self, x):
        # Fuzzificação
        x_exp = x.unsqueeze(2)
        mu_exp = self.mu.unsqueeze(0)
        sigma_exp = self.sigma.unsqueeze(0)

        # Função de pertinência Gaussiana
        membership = torch.exp(-0.5 * torch.pow((x_exp - mu_exp) / (sigma_exp + 1e-6), 2))

        # Ativação das Regras (T-norma do produto)
        rule_act = torch.prod(membership, dim=1)

        # Normalização
        norm_act = rule_act / (torch.sum(rule_act, dim=1, keepdim=True) + 1e-8)

        # Defuzzificação (Logit)
        return self.rule_weights(norm_act)

# 3. TREINAMENTO OTIMIZADO
model = NeuroFuzzyModel(input_dim=4, n_rules=20)
# pos_weight ajuda a focar nos inadimplentes (minoria)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([15.0]))
optimizer = optim.Adam(model.parameters(), lr=0.05)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

print("Iniciando Treinamento Neuro-Fuzzy...")
for epoch in range(101):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()
    scheduler.step()

    if epoch % 20 == 0:
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            print(f"Epoch {epoch:3} | Loss: {loss.item():.4f} | Std Dev Probs: {probs.std():.4f}")

# 4. GERAÇÃO DE RESULTADOS (TESTING)
model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
    test_logits = model(X_test_t)
    final_probs = torch.sigmoid(test_logits).numpy().flatten()

# Salvar
submission = pd.DataFrame({
    'Id': range(1, len(final_probs) + 1),
    'Probability': final_probs
})
submission.to_csv('results_neuro_fuzzy.csv', index=False)

Iniciando Treinamento Neuro-Fuzzy...
Epoch   0 | Loss: 1.3519 | Std Dev Probs: 0.0051
Epoch  20 | Loss: 1.2791 | Std Dev Probs: 0.0972
Epoch  40 | Loss: 1.2671 | Std Dev Probs: 0.1237
Epoch  60 | Loss: 1.2619 | Std Dev Probs: 0.1199
Epoch  80 | Loss: 1.2611 | Std Dev Probs: 0.1179
Epoch 100 | Loss: 1.2606 | Std Dev Probs: 0.1181

Processo concluído! O arquivo 'submission_neuro_fuzzy.csv' foi gerado.
